<a href="https://colab.research.google.com/github/Azlan-Qaisrani/my-first/blob/main/PDFtoJSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai pdfminer.six

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 50.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import openai

api_key=userdata.get('openai')

if not api_key:
  raise ValueError('OpenAI API key not found')

client=openai.OpenAI(api_key=api_key)

In [9]:
from google.colab import files
uploaded=files.upload()
pdf_filename=list(uploaded.keys())[0]
print(f'Uploaded file: {pdf_filename}')

Saving SampleHealthReport.pdf to SampleHealthReport.pdf
Uploaded file: SampleHealthReport.pdf


Define the Function Schema

The function schema is like a contract or blueprint that tells GPT:

“You are not just generating text. You are calling a function named extract_patient_info, and here’s what the input should look like.”

It's based on JSON Schema standards and includes:

"name" The name of the virtual function GPT is calling
"description" Helps the model understand what the function is for
"parameters" Specifies the expected structure of the output (a JSON object)
"properties" Lists all fields we want from the report
"required" Tells GPT which fields are mandatory




In [10]:
patient_info_function = {
    "name": "extract_patient_info",
    "description": "Extracts key personal fields from a health report.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "patient_id": {"type": "string"},
            "mrn": {"type": ["string", "null"]},
            "birth_date": {"type": "string", "format": "date"},
            "age": {"type": "integer"},
            "gender": {"type": "string"},
            "address": {"type": "string"},
            "phone": {
                "type": "object",
                "properties": {
                    "home": {"type": ["string", "null"]},
                    "work": {"type": ["string", "null"]},
                    "cell": {"type": ["string", "null"]}
                }
            },
            "email": {"type": "string"},
            "provider": {"type": "string"},
            "referring_provider": {"type": "string"},
            "print_date": {"type": "string", "format": "date"}
        },
        "required": ["name", "patient_id", "birth_date", "age", "gender", "address", "email"]
    }
}

In [11]:
from pdfminer.high_level import extract_text

def extract_text_from_pdf(path: str) -> str:
    return extract_text(path)

pdf_text = extract_text_from_pdf(pdf_filename)
print("✅ PDF text extracted. Sample preview:\n")
print(pdf_text[:1000])

✅ PDF text extracted. Sample preview:

PATIENT CHART  
HONOUR, EDWARD
123 S MAIN STREET CHICAGO IL  60611 
O: (312) 555-1212(preferred)

DOB: 8/15/1966  AGE: 58 yrs.  Acct#: 148

DEMOGRAPHICS

NAME: HONOUR, EDWARD
PATIENT ID/#: 148
MRN:
BIRTH DATE: 8/15/1966
AGE: 58 yrs.
GENDER: M
ADDRESS: 123 S MAIN STREET

CHICAGO IL  60611

Home:
Work:
Cell: (202) 555-1212
EMAIL: ED@EXAMPLE.COM
PROVIDER: SMITH, JOHN, MD
REFERRING PROVIDER: JOHN SMITH
STE A
3196 E ELM STREET
CHICAGO IL  60611
(773) 555-1212

ALLERGIES

No data on file

MEDICATION DETAIL

Current:
SIG: epinephrine 0.15 mg/0.3 mL injection auto-injector, 0 days, Dispense #2 Each, 0 Refills, Directions: administer 0.15mg 
subcutaneously as needed
8/8/2023 SMITH,JOHN,MD
----------------------------------
Current:
SIG: lisinopril 20 mg oral tablet, 90 days, Dispense #90 Tablet, 0 Refills, Directions: Pt to take 1 tablet po a day
2/9/2024 SMITH,JOHN,MD
----------------------------------
Current:
SIG: finasteride 1 mg oral tablet, 60 days, 

In [13]:
import json

def call_extraction(text: str, function_schema: dict) -> dict:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You are a medical data extractor."},
            {"role": "user", "content": text}
        ],
        functions=[function_schema],
        function_call={"name": function_schema["name"]}
    )
    args = response.choices[0].message.function_call.arguments
    return json.loads(args)

try:
    patient_info = call_extraction(pdf_text, patient_info_function)
    print("✅ Patient Info Extracted:\n")
    print(json.dumps(patient_info, indent=2))
except Exception as e:
    print("❌ Extraction failed:", str(e))

❌ Extraction failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
